# 🐼 Pandas for Auditors — Course

**Goal**: discover and *experiment* with the most classic functions of **pandas**
(the Python data-manipulation library) starting from what you already know in Excel.

**Audience**: banking auditors, comfortable with Excel (auto-sums, `VLOOKUP`),
but **not Python specialists**.

**How to read this notebook**
- Every grey cell is *code*. Click inside it then press **`Shift` + `Enter`** to run it.
- **Run the cells in order, from top to bottom.** A cell may depend on the previous ones.
- For each concept, a callout reminds you of the **Excel equivalent**:

> 💡 **Excel equivalent**: what this corresponds to in a spreadsheet.

> ℹ️ **No real data** is used. The transaction data is *randomly generated*
> in this notebook; you can therefore run everything and break everything without risk.


## Table of contents

1. Getting started & vocabulary (`DataFrame`)
2. Generating a demo transaction dataset
3. Reading & writing files (`read_csv`, `read_excel`)
4. Looking at your data (`head`, `info`, `describe`)
5. Selecting columns and rows (`loc`, `iloc`)
6. Filtering (the equivalent of Excel *filters*)
7. Sorting (`sort_values`)
8. Creating / computing columns
9. Aggregating: `sum`, `mean`, `count` (≈ `SUM`, `AVERAGE`, `COUNT`)
10. Grouping: `groupby` (≈ `SUMIF`)
11. Cross-tabulations: `pivot_table` (≈ *pivot table*)
12. Joining two tables: `merge` (≈ `VLOOKUP`)
13. Working with dates
14. Data quality: missing values & duplicates
15. 🔎 Concrete audit cases (mini analytical review)
16. Exporting your results
17. Python lists: creation, manipulation, comprehensions
18. Reading a PDF and extracting data with PyMuPDF


## 1. Getting started & vocabulary

First we *import* the two tools we need. This is the equivalent of "opening Excel"
before you start.

- `pandas`: to manipulate tables of data. It is traditionally given the nickname `pd`.
- `numpy`: to generate numbers (we only use it to build the demo dataset). Nickname `np`.

> 💡 **Excel equivalent**: launching Excel. Here, we load the features once.


In [ ]:
import pandas as pd
import numpy as np

# Display all columns without truncating them
pd.set_option("display.max_columns", None)

print("pandas version:", pd.__version__)
print("All set ✅")

### The most important word: `DataFrame`

A **`DataFrame`** is quite simply **a table**: **columns** (with a name) and **rows**
(numbered from 0). It is the exact equivalent of **an Excel sheet** or a **structured table**.

| pandas vocabulary | Excel equivalent |
|---|---|
| `DataFrame` | a sheet / a table |
| a `column` | a column (A, B, C…) but named |
| a `row` | a row |
| the `index` | the row number (starts at **0**) |
| a `Series` | a single isolated column |

A small toy example to visualise:


In [ ]:
example = pd.DataFrame({
    "account":  ["A001", "A002", "A003"],
    "amount":   [1500, 320, 9800],
    "currency": ["EUR", "EUR", "USD"],
})

example

Notice the column on the far left (0, 1, 2): it is the **index**, the equivalent of the row number.
⚠️ In Python, **we count from 0**, not from 1.

## 2. Generating a demo transaction dataset

We build ~500 fictitious banking transactions. **You don't need to understand this code**:
simply run it. In real life, this data would come from a core-banking export
or from an Excel file (see section 3).

A few realistic "traps" have been **deliberately slipped** into the data (duplicates,
missing values, round amounts, amounts just below a threshold…) for the audit cases in section 15.


In [ ]:
np.random.seed(42)   # makes the draw reproducible: everyone gets the same data
n = 500

dates = pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n), unit="D")

accounts = ["FR7610011000601234567890185", "FR7630004000031234567890143",
            "FR7612548029981234567890161", "FR7620041010051234567890138"]

counterparties = ["ALPHA SARL", "BETA SA", "GAMMA GMBH", "DELTA LTD",
                  "EPSILON SAS", "ZETA BV", "Individual", "ETA TRADING"]

df = pd.DataFrame({
    "transaction_id":   range(1, n + 1),
    "date":             dates,
    "account_id":       np.random.choice(accounts, n),
    "counterparty":     np.random.choice(counterparties, n),
    "transaction_type": np.random.choice(["Transfer", "Direct Debit", "Card", "Cash", "Check"],
                                         n, p=[0.40, 0.20, 0.20, 0.10, 0.10]),
    "direction":        np.random.choice(["Debit", "Credit"], n, p=[0.6, 0.4]),
    "channel":          np.random.choice(["Online", "Branch", "ATM", "Mobile"], n),
    "amount":           np.round(np.random.lognormal(mean=6.5, sigma=1.2, size=n), 2),
    "currency":         np.random.choice(["EUR", "USD", "GBP"], n, p=[0.85, 0.10, 0.05]),
    "country":          np.random.choice(["LU", "FR", "DE", "BE", "US", "GB"], n),
})

# --- deliberate traps for the audit cases ---
# "round" amounts
df.loc[np.random.choice(df.index, 15, replace=False), "amount"] = \
    np.random.choice([1000, 5000, 10000, 50000], 15)
# amounts just BELOW the 10,000 reporting threshold (structuring)
df.loc[np.random.choice(df.index, 8, replace=False), "amount"] = \
    np.random.choice([9900.0, 9950.0, 9800.0, 9990.0], 8)
# missing values
df.loc[np.random.choice(df.index, 12, replace=False), "counterparty"] = np.nan
df.loc[np.random.choice(df.index, 7,  replace=False), "country"] = np.nan
# duplicates (same transaction recorded twice, with a different id)
dups = df.sample(6, random_state=1).copy()
dups["transaction_id"] = range(n + 1, n + 1 + len(dups))
df = pd.concat([df, dups], ignore_index=True)

# we shuffle the row order to make it realistic
df = df.sample(frac=1, random_state=7).reset_index(drop=True)

print("Dataset ready:", df.shape[0], "rows,", df.shape[1], "columns")

**Meaning of the columns**

| Column | Description |
|---|---|
| `transaction_id` | unique identifier of the operation |
| `date` | date of the operation |
| `account_id` | IBAN of the bank account |
| `counterparty` | counterparty (originator / beneficiary) |
| `transaction_type` | type of operation |
| `direction` | Debit or Credit |
| `channel` | channel (Online, Branch, ATM, Mobile) |
| `amount` | amount of the operation |
| `currency` | currency |
| `country` | country of the counterparty |


## 3. Reading & writing files

In practice, your data arrives in a **file** (CSV or Excel). For the demo, we first save
our dataset to two files, then show how to **read them back**.

> 💡 **Excel equivalent**: *File → Save As* (writing) and *File → Open* (reading).


In [ ]:
# Write (export) -- index=False so we don't write the row-number column
df.to_csv("transactions.csv", index=False)
df.to_excel("transactions.xlsx", index=False)   # requires the openpyxl package
print("Files transactions.csv and transactions.xlsx created.")

In [ ]:
# Read (import) a CSV
df_csv = pd.read_csv("transactions.csv")

# Read an Excel file
df_xlsx = pd.read_excel("transactions.xlsx")

print("Rows read from the CSV   :", len(df_csv))
print("Rows read from the Excel :", len(df_xlsx))

> ℹ️ If `read_excel` / `to_excel` returns an error saying **openpyxl** is missing, run once,
> in a cell, the command: `!pip install openpyxl` (the exclamation mark launches an installation).

For your real files, simply replace the name:
`pd.read_excel("C:/Users/me/Desktop/my_export.xlsx")`.
The other sections continue with the `df` variable.


## 4. Looking at your data

An auditor's first reflex: **look** at what you have on hand.

> 💡 **Excel equivalent**: scrolling through the first rows, looking at the header, counting the rows.


In [ ]:
df.head()        # the first 5 rows (head). df.head(10) for 10 rows.

In [ ]:
df.tail(3)       # the last 3 rows (tail)

In [ ]:
df.shape         # (number of rows, number of columns)

In [ ]:
df.columns       # the list of column names

In [ ]:
df.info()        # type of each column + number of non-missing values

`df.describe()` computes the **statistics** of the numeric columns in one go
(count, mean, standard deviation, min, quartiles, max).

> 💡 **Excel equivalent**: `COUNT`, `AVERAGE`, `MIN`, `MAX`, `STDEV`… all in one line.


In [ ]:
df.describe()

## 5. Selecting columns and rows

### One or several columns

> 💡 **Excel equivalent**: selecting the "amount" column, or the "amount" and "currency" columns.


In [ ]:
df["amount"].head()                       # ONE column (in brackets, its name in quotes)

In [ ]:
df[["amount", "currency"]].head()          # SEVERAL columns: a list [ ... ] of names

### Specific rows with `.loc` and `.iloc`

- `.iloc[...]`: selection by **position** (i as in *integer*, the number). We count from 0.
- `.loc[...]`: selection by index **label** and column **name**.

> 💡 **Excel equivalent**: going to a specific cell/range, for example `B2:C5`.


In [ ]:
df.iloc[0]              # the very first row (position 0)

In [ ]:
df.iloc[0:5]            # the first 5 rows (positions 0 to 4)

In [ ]:
# .loc with a column name: the date + amount columns of the first 5 rows
df.loc[0:4, ["date", "amount"]]

## 6. Filtering rows (Excel *filters*)

This is probably the most useful operation in audit: **keeping only the rows that meet
a condition**.

The principle: you write a **condition** in brackets, and pandas keeps only the rows where it is true.

> 💡 **Excel equivalent**: *auto-filters*, or the `FILTER()` function.


In [ ]:
# All transactions over 10,000
df[df["amount"] > 10000].head()

In [ ]:
# All cash operations ( == means "is equal to")
df[df["transaction_type"] == "Cash"].head()

### Combining several conditions

- `&` means **AND** (all conditions true)
- `|` means **OR** (at least one true)
- ⚠️ Each condition must be in **parentheses**.


In [ ]:
# Cash AND amount greater than 5,000
df[(df["transaction_type"] == "Cash") & (df["amount"] > 5000)].head()

In [ ]:
# Country = US OR GB
df[df["country"].isin(["US", "GB"])].head()       # isin = "is part of this list"


In [ ]:
# Amount between 9,000 and 10,000
df[df["amount"].between(9000, 10000)].head()

In [ ]:
# The counterparty name contains "SA" (text search)
df[df["counterparty"].str.contains("SA", na=False)].head()

> ℹ️ `na=False` tells pandas to ignore missing values during the text search,
> otherwise they would raise an error.

To **count** how many rows match a filter, chain with `.shape[0]` or `len(...)`:


In [ ]:
nb = len(df[df["amount"] > 10000])
print("Transactions > 10,000:", nb)

## 7. Sorting

> 💡 **Excel equivalent**: *Data → Sort* (ascending / descending).


In [ ]:
# Sort by amount DESCENDING (the largest at the top)
df.sort_values("amount", ascending=False).head()

In [ ]:
# Sort on two columns: first by account, then by date
df.sort_values(["account_id", "date"]).head()

## 8. Creating / computing a new column

You create a column by writing `df["new_name"] = ...`.

> 💡 **Excel equivalent**: adding a column with a formula that copies down all rows.


In [ ]:
# Convert all amounts to EUR (fictitious rates, for the example)
rates = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17}

df["eur_rate"]   = df["currency"].map(rates)     # map = associate each currency with its rate
df["amount_eur"] = (df["amount"] * df["eur_rate"]).round(2)

df[["amount", "currency", "eur_rate", "amount_eur"]].head()

In [ ]:
# A column based on a condition: flag the "large" amounts
df["large_amount"] = df["amount_eur"] > 10000      # gives True / False
df[["amount_eur", "large_amount"]].head()

## 9. Aggregating: `sum`, `mean`, `count`, `min`, `max`

You apply a computation to **a whole column**.

> 💡 **Excel equivalent**: `=SUM(...)`, `=AVERAGE(...)`, `=COUNT(...)`, `=MIN(...)`, `=MAX(...)`.


In [ ]:
print("Total of amounts (EUR) :", df["amount_eur"].sum().round(2))
print("Average amount (EUR)    :", df["amount_eur"].mean().round(2))
print("Median amount (EUR)     :", df["amount_eur"].median().round(2))
print("Largest amount (EUR)    :", df["amount_eur"].max())
print("Number of transactions  :", df["amount_eur"].count())

`value_counts()` counts how many times each value appears: perfect for a categorical column.

> 💡 **Excel equivalent**: `COUNTIF` repeated for each value, or a pivot table in count mode.


In [ ]:
df["transaction_type"].value_counts()

## 10. Grouping: `groupby` ≈ `SUMIF`

`groupby` = "**for each** category, compute…". It is the key tool for **summaries**.

> 💡 **Excel equivalent**: `SUMIF`, or a simple **pivot table**.

Reading the code below: *"for each `transaction_type`, take the `sum` of `amount_eur`"*.


In [ ]:
df.groupby("transaction_type")["amount_eur"].sum().round(2)

In [ ]:
# We can sort the result from largest to smallest
df.groupby("transaction_type")["amount_eur"].sum().round(2).sort_values(ascending=False)

We can group on **several** columns, and ask for **several** computations at once with `.agg(...)`:

In [ ]:
# For each account: total, average and number of operations
df.groupby("account_id")["amount_eur"].agg(["sum", "mean", "count"]).round(2)

In [ ]:
# Group on two levels: account then direction (Debit/Credit)
df.groupby(["account_id", "direction"])["amount_eur"].sum().round(2)

## 11. Cross-tabulations: `pivot_table`

When you want one category **in rows** and another **in columns**, this is exactly Excel's
**pivot table**.

> 💡 **Excel equivalent**: *Insert → Pivot Table*.

Here: `transaction_type` in rows, `direction` in columns, and the **sum** of amounts in the cells.


In [ ]:
pd.pivot_table(
    df,
    index="transaction_type",   # the rows
    columns="direction",        # the columns
    values="amount_eur",        # the value to aggregate
    aggfunc="sum",              # the computation: "sum", "mean", "count"...
    fill_value=0,               # replace empty cells with 0
).round(2)

In [ ]:
# Same thing but COUNTING the number of operations rather than summing
pd.pivot_table(df, index="channel", columns="direction",
               values="transaction_id", aggfunc="count", fill_value=0)

## 12. Joining two tables: `merge` ≈ `VLOOKUP`

Very common in audit: you have a table of operations, and **another table** of reference
(e.g. the characteristics of the accounts). You want to **bring** the info from the 2nd table into the 1st,
relying on a **common column** (here `account_id`).

> 💡 **Excel equivalent**: `VLOOKUP` (or `XLOOKUP`).

Let's first create a small reference table of the accounts:


In [ ]:
accounts_ref = pd.DataFrame({
    "account_id":  accounts,    # same variable as in section 2
    "holder":      ["Alpha Holding", "Beta Industries", "Gamma Trust", "Delta Family Office"],
    "segment":     ["Corporate", "Corporate", "Private", "Private"],
    "risk_level":  ["Low", "Medium", "High", "Medium"],
})
accounts_ref

We now **merge** `df` with `accounts_ref` on the common column `account_id`.

- `on="account_id"`: the join column (the "key" of the `VLOOKUP`).
- `how="left"`: we keep **all** the rows of `df` (the left table) — like a classic `VLOOKUP`
  that starts from the main table.


In [ ]:
df_enriched = df.merge(accounts_ref, on="account_id", how="left")

df_enriched[["transaction_id", "account_id", "holder", "segment", "risk_level", "amount_eur"]].head()

Now that we have the `segment` and the `risk_level`, we can build summaries on them —
this is where `merge` + `groupby` become powerful:

In [ ]:
# Total amount by risk level
df_enriched.groupby("risk_level")["amount_eur"].sum().round(2).sort_values(ascending=False)

## 13. Working with dates

The `date` column is already recognised as a real date (*datetime* type). We can then easily
extract the year, the month, the day of the week… via the `.dt` accessor.

> 💡 **Excel equivalent**: `YEAR()`, `MONTH()`, `WEEKDAY()`.


In [ ]:
# Check that the column really is a date
df["date"].dtype

In [ ]:
df["year"]     = df["date"].dt.year
df["month"]    = df["date"].dt.month            # 1 to 12
df["weekday"]  = df["date"].dt.dayofweek        # 0 = Monday ... 6 = Sunday
df["day_name"] = df["date"].dt.day_name()

df[["date", "year", "month", "weekday", "day_name"]].head()

**Monthly trend** of amounts — a classic of analytical review:

In [ ]:
monthly_trend = df.groupby(df["date"].dt.to_period("M"))["amount_eur"].sum().round(2)
monthly_trend

In [ ]:
# A small bar chart (optional but telling)
monthly_trend.plot(kind="bar", figsize=(10, 3), title="Total amount per month (EUR)");

## 14. Data quality: missing values & duplicates

An essential audit step: **making the data reliable** before analysing it.

### Missing values (`NaN` = *Not a Number*, an empty cell)

> 💡 **Excel equivalent**: spotting empty cells, `COUNTBLANK`.


In [ ]:
# How many missing values per column?
df.isna().sum()

In [ ]:
# See the rows where the counterparty is missing
df[df["counterparty"].isna()].head()

In [ ]:
# Two ways to handle it:
df_no_na  = df.dropna(subset=["counterparty"])           # drop those rows
df_filled = df.fillna({"counterparty": "UNKNOWN",
                       "country": "UNKNOWN"})             # replace the empty cell

print("Rows after dropna :", len(df_no_na))
print("Missing after fillna :", df_filled[["counterparty", "country"]].isna().sum().sum())

### Duplicates

> 💡 **Excel equivalent**: *Data → Remove Duplicates*, or a conditional formatting
> "duplicate values".

`duplicated()` flags duplicate rows. We can look for duplicates on **everything** or on a
**subset of business columns** (same date, same account, same amount, same counterparty…).


In [ ]:
# "Business" duplicates: same date, account, counterparty, type and amount
keys = ["date", "account_id", "counterparty", "transaction_type", "amount"]

duplicates = df[df.duplicated(subset=keys, keep=False)]    # keep=False -> keep ALL occurrences
print("Rows involved in a duplicate:", len(duplicates))
duplicates.sort_values(keys).head(10)

In [ ]:
# Get a table WITHOUT duplicates (keep only the 1st occurrence)
df_dedup = df.drop_duplicates(subset=keys, keep="first")
print("Before:", len(df), "-> After:", len(df_dedup))

## 15. 🔎 Concrete audit cases

We now combine everything above for a few typical **analytical tests**.
⚠️ These are simplified *teaching examples*, not a complete control methodology.


### 15.1 — The 10 largest amounts
Spotting the most significant operations.

In [ ]:
df.sort_values("amount_eur", ascending=False).head(10)[
    ["transaction_id", "date", "account_id", "counterparty", "amount_eur", "transaction_type"]
]

### 15.2 — "Round" amounts
Exactly round amounts (multiples of 1,000) may deserve particular attention.

In [ ]:
round_amounts = df[df["amount"] % 1000 == 0]
print("Operations with a round amount:", len(round_amounts))
round_amounts[["transaction_id", "date", "counterparty", "amount", "transaction_type"]].head(10)

### 15.3 — Amounts just below a threshold (*structuring*)
Operations between 9,000 and 9,999, i.e. **just below** the 10,000 reporting threshold:
a classic structuring pattern to watch.

In [ ]:
below_threshold = df[(df["amount"] >= 9000) & (df["amount"] < 10000)]
print("Operations just below 10,000:", len(below_threshold))
below_threshold[["transaction_id", "date", "account_id", "counterparty", "amount"]].sort_values("amount", ascending=False)

### 15.4 — Weekend operations
Activity on a Saturday/Sunday: sometimes unusual depending on the context.

In [ ]:
weekend = df[df["date"].dt.dayofweek >= 5]      # 5 = Saturday, 6 = Sunday
print("Weekend operations:", len(weekend))
weekend[["transaction_id", "date", "day_name", "counterparty", "amount_eur"]].head(10)

### 15.5 — Significant cash
Cumulative cash operations, by account, above a materiality threshold.

In [ ]:
cash_ops = df[(df["transaction_type"] == "Cash") & (df["amount_eur"] > 3000)]
cash_ops.groupby("account_id")["amount_eur"].agg(["count", "sum"]).round(2).sort_values("sum", ascending=False)

## 16. Exporting your results

Once a sample or a summary is obtained, we export it to Excel/CSV to share it
or document it in the working file.

> 💡 **Excel equivalent**: *Save As*.


In [ ]:
# We export the table of significant international transfers (section 15)
local_zone = ["LU", "FR", "DE", "BE"]
suspicious = df[
    (df["transaction_type"] == "Transfer")
    & (df["amount_eur"] > 8000)
    & (~df["country"].isin(local_zone))
].sort_values("amount_eur", ascending=False)

suspicious.to_excel("operations_to_review.xlsx", index=False)
print("File 'operations_to_review.xlsx' created with", len(suspicious), "rows.")

## 17. Python lists: creation, manipulation, comprehensions

Before working with PDFs (section 18), it is useful to master **Python lists** —
the basic data structure found everywhere: column names, country codes,
text-extraction results…

> 💡 **Excel equivalent**: a range of values in a single column, or a comma-separated
> enumeration of values in a formula (`SUMIFS(…;"FR";"LU")`).


### 17.1 — Creating and iterating over a list


In [ ]:
# Create a list (in brackets, values separated by commas)
countries = ['France', 'Luxembourg', 'Panama', 'Cyprus', 'Malta', 'Singapore']
print(countries)
print("Number of elements:", len(countries))

In [ ]:
# Access by position (the index starts at 0)
print(countries[0])    # first element
print(countries[-1])   # last element
print(countries[1:4])  # slice: index 1 included up to 4 excluded (Lux, Pan, Cyprus)

In [ ]:
# Check for the presence of a value
print('Panama' in countries)    # True
print('Germany' in countries)   # False
print(countries.index('Cyprus'))  # position of 'Cyprus' in the list

### 17.2 — Adding, removing, sorting


In [ ]:
countries_work = countries.copy()  # we work on a copy so as not to modify the original

countries_work.append('Malta')              # add ONE element at the end
countries_work.extend(['Germany', 'Belgium'])  # add SEVERAL elements
print("After additions:", countries_work)

countries_work.remove('Malta')   # remove the FIRST occurrence of a value
removed = countries_work.pop()   # remove AND retrieve the LAST element
print("Removed:", removed)
print("After removals:", countries_work)

In [ ]:
# Sort
country_copies = ['Panama', 'France', 'Luxembourg', 'Cyprus']

country_copies.sort()                 # alphabetical sort IN PLACE (modifies the list)
print("sort() in place:", country_copies)

country_original = ['Panama', 'France', 'Luxembourg', 'Cyprus']
country_sorted = sorted(country_original)    # sort WITHOUT modifying the original
print("sorted() without modification:", country_original)
print("New sorted list             :", country_sorted)

### 17.3 — List comprehensions

**Comprehensions** let you create or filter a list in a single elegant line.
It is one of the most useful constructs in Python.

```
[expression   for element in iterable   if condition]
```


In [ ]:
countries = ['France', 'Luxembourg', 'Panama', 'Cyprus', 'Malta', 'Singapore']

# Transform each element
countries_upper = [p.upper() for p in countries]
print("Uppercase:", countries_upper)

# Filter on a condition
long_countries = [p for p in countries if len(p) > 6]
print("Names > 6 characters:", long_countries)

# Combine transformation AND filter
codes = [p[:2].upper() for p in countries if p != 'Singapore']
print("2-letter codes (excluding Singapore):", codes)

In [ ]:
# Audit case: filter amounts above a threshold
amounts = [1200.0, 9800.0, 450.50, 7500.0, 3300.0, 9950.0, 150.0]

large_ones = [m for m in amounts if m > 5000]
print("Amounts > 5,000:", large_ones)

# Count directly (sum over a list of booleans)
nb_suspects = sum(1 for m in amounts if 9000 <= m < 10000)
print("Amounts between 9,000 and 9,999:", nb_suspects)

### 17.4 — Lists and pandas

Lists and pandas often work together.

> 💡 A pandas `Series` *is* an enriched list (index, name, vectorised methods).


In [ ]:
import pandas as pd

# List → pandas Series
country_list = ['France', 'Luxembourg', 'Panama', 'Cyprus']
s = pd.Series(country_list, name='country')
print(s)

# Series → Python list
back = s.tolist()
print("\nBack to a list:", back)

# A DataFrame's column names ARE a list
df_example = pd.DataFrame({'a': [1, 2], 'b': [3, 4], 'c': [5, 6]})
columns = df_example.columns.tolist()
print("\nColumns:", columns)

In [ ]:
# Filtering with .isin(): equivalent to a comprehension but on a whole column
import numpy as np
np.random.seed(42)
df_tx = pd.DataFrame({
    'country': np.random.choice(['France', 'Panama', 'Cyprus', 'Germany', 'Malta'], 10),
    'amount': np.random.randint(500, 15000, 10),
})

risk_country = ['Panama', 'Cyprus', 'Malta']   # a Python list...
alerts = df_tx[df_tx['country'].isin(risk_country)]  # ...used in .isin()
print("Operations to high-risk countries:\n", alerts)

## 18. Reading a PDF and extracting data with PyMuPDF

In audit, documents often arrive in **PDF** format: inspection reports,
account statements, transfer confirmations… PyMuPDF (imported as `fitz`) lets you
extract the text, then search it for structured data (dates, country codes…).

> ℹ️ **Installation**: if the module is not available, run once:
> `!pip install pymupdf`

> 💡 **Excel equivalent**: the *Data → From Text/PDF File* feature of Excel 365,
> but here in code — which lets you process dozens of files in a loop.


### 18.1 — Generating a demo PDF

We first create a fake "banking correspondence statement" to have a working file.
In practice, you will receive the PDF directly from your client.


In [ ]:
import fitz  # PyMuPDF

pdf_content = [
    "Banking Correspondence Statement - Financial Year 2024",
    "",
    "Prepared by: Compliance Department  |  Issue date: 15/04/2024",
    "Period covered: 01/01/2024 to 31/03/2024",
    "",
    "Flows recorded:",
    "",
    "  1.  Transfer received on 08/01/2024 | Origin: France | Amount: 12 500 EUR",
    "  2.  Transfer sent on 22/01/2024 | Destination: Luxembourg | Amount: 8 900 EUR",
    "  3.  Settlement received on 03/02/2024 | Origin: Panama | Amount: 45 000 USD",
    "  4.  Transfer sent on 14/02/2024 | Destination: Cyprus | Amount: 9 950 EUR",
    "  5.  Transfer received on 27/02/2024 | Origin: Germany | Amount: 3 200 EUR",
    "  6.  Settlement sent on 05/03/2024 | Destination: Malta | Amount: 7 800 EUR",
    "  7.  Transfer received on 19/03/2024 | Origin: France | Amount: 22 000 EUR",
    "  8.  Transfer sent on 28/03/2024 | Destination: Singapore | Amount: 15 000 USD",
    "",
    "Total inbound flows  : 82 700 EUR",
    "Total outbound flows : 41 650 EUR",
    "",
    "Observations: flow #3 (Panama) and #4 (Cyprus) flagged for AML-CFT review.",
    "Next review scheduled for 30/06/2024.",
]

doc = fitz.open()          # create an empty document
page = doc.new_page()      # add a page
text = "\n".join(pdf_content)
page.insert_text((60, 60), text, fontsize=10.5)
doc.save("correspondence_statement.pdf")
doc.close()
print("PDF created: correspondence_statement.pdf")

### 18.2 — Reading and extracting the text


In [ ]:
doc = fitz.open("correspondence_statement.pdf")

print(f"Number of pages: {doc.page_count}")
print("─" * 60)

# Extract the text from each page
full_text = ""
for page_num, page in enumerate(doc, start=1):
    page_text = page.get_text()            # all the text of the page
    full_text += page_text
    print(f"── Page {page_num} ──")
    print(page_text[:300])                 # preview

doc.close()

### 18.3 — Extracting dates with a regular expression (*regex*)

We use the `re` module to look for all the **`DD/MM/YYYY`** patterns in the text.


In [ ]:
import re

# Regex pattern: two digits, slash, two digits, slash, four digits
date_pattern = r'\d{2}/\d{2}/\d{4}'

dates_found = re.findall(date_pattern, full_text)
print("Dates found:", dates_found)

# Convert to pandas datetime objects so we can sort/filter them
dates_pd = pd.to_datetime(dates_found, format='%d/%m/%Y')
print("\npandas dates:", dates_pd.tolist())

### 18.4 — Extracting country names by matching against a list

We have a **reference list** of countries. We look for which ones appear in the text.
This is a direct use of the list comprehensions seen in section 17.


In [ ]:
reference_countries = [
    'France', 'Luxembourg', 'Panama', 'Cyprus', 'Malta',
    'Singapore', 'Germany', 'Belgium', 'Switzerland', 'Monaco',
    'United Arab Emirates', 'Cayman Islands',
]

# Comprehension: keep only the countries that appear in the text
countries_in_pdf = [p for p in reference_countries if p in full_text]
print("Countries mentioned in the PDF:", countries_in_pdf)

### 18.5 — Building a DataFrame from the extracted text

We combine regex + lists to reconstruct a structured table.


In [ ]:
# Regex to extract the flow lines (numbered 1. to 8.)
# Fields are separated by " | "; \s* keeps the pattern tolerant to spacing.
flow_pattern = r'(\d+)\.\s+(Transfer|Settlement)\s+(received|sent)\s+on\s+(\d{2}/\d{2}/\d{4})\s*\|\s*(?:Origin|Destination):\s*([\w ]+?)\s*\|\s*Amount:\s*([\d ]+\s+\w+)'

lines = re.findall(flow_pattern, full_text)
print("Extracted lines:")
for l in lines:
    print(" ", l)

In [ ]:
# Build the DataFrame
if lines:
    df_pdf = pd.DataFrame(lines, columns=[
        'num', 'flow_type', 'direction', 'date', 'country', 'raw_amount'
    ])
    df_pdf['date'] = pd.to_datetime(df_pdf['date'], format='%d/%m/%Y')
    # Clean the amount: remove spaces and separate the figure from the currency
    df_pdf['currency'] = df_pdf['raw_amount'].str.extract(r'([A-Z]{3})$')
    df_pdf['amount']   = df_pdf['raw_amount'].str.replace(r'[^\d]', '', regex=True).astype(float)
    df_pdf = df_pdf.drop(columns='raw_amount')
    df_pdf

In [ ]:
# Identify flows to high-risk countries (list from section 17)
risk_country = ['Panama', 'Cyprus', 'Malta', 'Singapore']

if lines:
    df_pdf['country_alert'] = df_pdf['country'].isin(risk_country)
    print("Flagged flows:")
    print(df_pdf[df_pdf['country_alert']][['date', 'flow_type', 'direction', 'country', 'amount', 'currency']])

## 🎓 Recap: from the spreadsheet to pandas

| Need | Excel | pandas |
|---|---|---|
| Open a file | *File → Open* | `pd.read_excel(...)` / `pd.read_csv(...)` |
| Look at the data | scroll | `df.head()`, `df.info()`, `df.describe()` |
| Filter | auto-filters | `df[df["col"] > x]` |
| Sort | *Data → Sort* | `df.sort_values("col")` |
| Computed column | copied-down formula | `df["new"] = ...` |
| Conditional sum | `SUMIF` | `df.groupby("cat")["val"].sum()` |
| Pivot table | pivot table | `pd.pivot_table(...)` |
| Lookup/join | `VLOOKUP` | `df.merge(other, on="key")` |
| Read/write a list | `[a, b, c]`, `append`, `pop`… | Python list |
| List comprehension | `[x for x in l if cond]` | Python list |
| Read a PDF | `fitz.open(...)` / `.get_text()` | PyMuPDF (`fitz`) |
| Extract data (regex) | `re.findall(pattern, text)` | `re` module |
| Remove duplicates | *Remove Duplicates* | `df.drop_duplicates()` |
| Empty cells | manual spotting | `df.isna()`, `df.fillna()`, `df.dropna()` |
| Save | *Save As* | `df.to_excel(...)` / `df.to_csv(...)` |

---

> Sections 17 and 18 cover **Python lists** and **PDF reading** (PyMuPDF).

**To go further**: change the thresholds of the audit cases, change the columns of the `groupby`,
test your own filters. The best way to learn pandas is to **break then fix** 🙂.
